 # 01_C0.py : LOOCV for C0, C0+ and C0n (run this first)

 * C0  : supervised baseline.

 * C0+ : baseline + the same extra labelled epochs as one pseudo round (fresh restart from the baseline for each round,

         same labelled draws) -> "is it just more training?" control.

 * C0n : supervised with noise augmentation estimated from Logger B.



 Preprocessing = z-score, evaluation = stitched windows (see thesis_common.py).

 Saves, per (fold, seed): `units_C0/fXX_sY.pkl` (rows + baseline weights + C0+ control rows, used by the C1/C2/C3 scripts),

 `models/C0/fXX_sY_{C0,C0plus_rN,C0n}.pth`, and the LOOCV splits in `fold_splits.json`.

 Resumable: finished units are skipped on rerun.

In [2]:
from thesis_common import *

init_data(unlab=("B",))                 # labelled data + Logger B (only needed for the noise floor)
SP = build_splits(); save_splits(SP)
print(f"{len(SP)} folds saved to {OUT / 'fold_splits.json'}")
for fi, s in SP.items(): print(f"fold {fi+1}: test={s['test_ind']} | val={s['val_inds']} | n_train_recordings={len(s['train'])}")


loaded (z-score): L 76.4 h | B 1650.4 h
14 folds saved to Model_results/LOOCV_zscore_stitched_final/fold_splits.json
fold 1: test=CC-07-107_11-02-2019 | val=['CC-07-59_20-05-2018', 'CC-07-48_22-05-2018'] | n_train_recordings=14
fold 2: test=CC-07-108_11-02-2019 | val=['CC-07-48_08-10-2018', 'CC-07-48_18-02-2019'] | n_train_recordings=14
fold 3: test=CC-07-115 | val=['CC-07-48_10-10-2018', 'CC-07-48_26-05-2018'] | n_train_recordings=13
fold 4: test=CC-07-47_14-02-2018 | val=['CC-07-48_22-05-2018', 'CC-07-48_18-02-2019'] | n_train_recordings=14
fold 5: test=CC-07-48_04-10-2018 | val=['CC-07-108_11-02-2019', 'CC-07-48_08-10-2018'] | n_train_recordings=14
fold 6: test=CC-07-48_06-10-2018 | val=['CC-07-48_04-10-2018', 'CC-07-115'] | n_train_recordings=13
fold 7: test=CC-07-48_08-04-2019 | val=['CC-07-48_06-10-2018', 'CC-07-48_26-05-2018'] | n_train_recordings=12
fold 8: test=CC-07-48_08-10-2018 | val=['CC-07-47_14-02-2018', 'CC-07-107_11-02-2019'] | n_train_recordings=14
fold 9: test=CC-07-

 ## Noise floor of Logger B vs labelled data (for C0n), estimated in z-score units

In [3]:
rngf = np.random.RandomState(0)
def win_starts(n, k): return rngf.randint(0, n - W, size=min(k, max(1, (n - W) // W)))

def noise_floor(mats, k=200, q=10):
    v = []
    for m in mats.values():
        for s in win_starts(len(m), k):
            d = np.diff(m[s:s + W], axis=0); mad = np.median(np.abs(d - np.median(d, 0)), 0) / (0.6745 * np.sqrt(2)); v.append(mad)
    return np.percentile(np.array(v), q, axis=0)

NF_ = {"L": noise_floor(matrices), "B": noise_floor(MATS["B"])}
nf_df = pd.DataFrame(NF_, index=["accX", "accY", "accZ", "gyrX", "gyrY", "gyrZ", "Pdiff"]).T; display(nf_df.round(4))
NOISE_SIGMA = np.sqrt(np.maximum(NF_["B"] ** 2 - NF_["L"] ** 2, 0)).astype(np.float32); NOISE_SIGMA[6] = 0.0
USE_NOISE_ARM = bool((NOISE_SIGMA > 1e-3).any())
print("extra noise sigma injected into labelled windows (C0n):", NOISE_SIGMA.round(4), "| C0n enabled:", USE_NOISE_ARM)
nf_df.to_csv(OUT / "noise_floor_by_source.csv"); json.dump(NOISE_SIGMA.tolist(), open(OUT / "noise_sigma.json", "w"))


,accX,accY,accZ,gyrX,gyrY,gyrZ,Pdiff
L,0.0073,0.0175,0.0198,0.0184,0.0137,0.0197,0.0025
B,0.0022,0.0045,0.0080,0.0042,0.0030,0.0032,0.0097


extra noise sigma injected into labelled windows (C0n): [0. 0. 0. 0. 0. 0. 0.] | C0n enabled: False


 ## One unit = (fold, seed): C0, C0+ (rounds 1..PS_ITERS), C0n

In [4]:
def run_c0_unit(fi, sp, seed):
    check_stop(); path = unit_path("C0", fi, seed)
    if path.exists(): return
    names_train = sp["train"]; ctx = dict(names_val=sp["val"], names_test=sp["test"])
    set_seed(SEED + 5000 + fi); Xv, Yv = LabelledSampler(sp["val"], SEED + 5000 + fi).batch(N_VAL)
    meta0 = dict(fold=fi + 1, test_ind=sp["test_ind"], seed=seed, frac=1.0, pool=None, sampling=None, pool_frac=None, exp="shared")
    print(f"\n=== fold {fi+1} seed {seed}: C0 baseline (test individual {sp['test_ind']}) ===")
    base = train_supervised(names_train, Xv, Yv, seed * 100 + fi); save_state(base, "C0", fi, seed, "C0")
    st = dict(rows=[], base_state=cpu_sd(base.state_dict()), base_info=base.info, ctrl_val={})
    st["rows"].append(row_of({**meta0, "arm": "C0", "cond": "C0", "round": 0, "kind": "baseline", **base.info}, base, ctx))
    print(f"C0 | val macro F1 {st['rows'][-1]['val_macro']:.3f} | test macro F1 {st['rows'][-1]['test_macro']:.3f} | {base.info}")
    for r in range(1, PS_ITERS + 1):
        c = finetune(base, names_train, 5000 + fi * 100 + r + seed * 7)
        row = row_of({**meta0, "arm": "C0+", "cond": "C0+", "round": r, "kind": "ctrl"}, c, ctx)
        st["rows"].append(row); st["ctrl_val"][r] = row; save_state(c, "C0", fi, seed, f"C0plus_r{r}"); del c
    if USE_NOISE_ARM:
        n_ = train_supervised(names_train, Xv, Yv, seed * 100 + fi, noise_sigma=NOISE_SIGMA); save_state(n_, "C0", fi, seed, "C0n")
        st["rows"].append(row_of({**meta0, "arm": "C0n", "cond": "C0n", "round": 0, "kind": "noise", **n_.info}, n_, ctx)); del n_
    atomic_dump(st, path)


 ## Run LOOCV

In [ ]:
try:
    for fi in sorted(SP):
        if fi % N_WORKERS != WORKER: continue
        for seed in SEEDS: run_c0_unit(fi, SP[fi], seed)
except StopRun:
    print("stopped cleanly; rerun this cell to resume")



=== fold 1 seed 0: C0 baseline (test individual CC-07-107_11-02-2019) ===
C0 | val macro F1 0.707 | test macro F1 0.643 | {'epochs_run': 77, 'best_val_loss': 0.4331658264947316}

=== fold 1 seed 1: C0 baseline (test individual CC-07-107_11-02-2019) ===


In [ ]:
done = sorted((OUT / "units_C0").glob("*.pkl")); print(f"{len(done)} C0 units finished of {len(SP) * len(SEEDS)}")
rows = [r for p in done for r in pickle.load(open(p, "rb"))["rows"]]
if rows:
    d = pd.DataFrame(rows).drop(columns="cm"); d.to_csv(OUT / "C0_family_results.csv", index=False)
    display(d.groupby("cond")[["val_macro", "test_macro", "test_macroNoOther", "test_target"]].mean().round(4))
